# 01 — Limpeza da lista de preços da CMED

**Projeto:** Preços de medicamentos no Brasil
**Fonte:** Lista de Preços de Medicamentos (PMC) — CMED/ANVISA, arquivo XLSX publicado mensalmente em
https://www.gov.br/anvisa/pt-br/assuntos/medicamentos/cmed/precos

**Objetivo deste notebook:** transformar o arquivo bruto da CMED em uma tabela limpa, com tipos corretos,
pronta para a análise exploratória (`02_eda.ipynb`).

Principais problemas do arquivo bruto:
- o cabeçalho real não está na primeira linha (há um bloco de notas explicativas acima dele);
- os preços vêm como texto, com vírgula decimal;
- existem dezenas de colunas de preço, uma para cada alíquota de ICMS;
- campos Sim/Não, placeholders como `-` e nomes de coluna com espaços e acentos.

In [2]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

PASTA_RAW = Path("../data/raw")
PASTA_PROCESSED = Path("../data/processed")
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Usa o arquivo da CMED mais recente que estiver em data/raw
arquivos = sorted(PASTA_RAW.glob("xls_conformidade*.xls*"))
ARQUIVO = arquivos[-1]
ARQUIVO

WindowsPath('../data/raw/xls_conformidade_site_20260909_222937320.xlsx')

## 1. Leitura do arquivo

Em vez de fixar a linha do cabeçalho, procuramos a linha que começa com `SUBSTÂNCIA`.
Assim o notebook continua funcionando se a CMED mudar o tamanho do bloco de notas nas próximas edições.

In [3]:
previa = pd.read_excel(ARQUIVO, header=None, nrows=100, usecols=[0])
linha_cabecalho = previa.index[previa[0].astype(str).str.strip() == "SUBSTÂNCIA"][0]

df_raw = pd.read_excel(ARQUIVO, header=linha_cabecalho, dtype=str)
print(f"Cabeçalho na linha {linha_cabecalho + 1} do Excel")
print(f"{df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
df_raw.head(3)

Cabeçalho na linha 42 do Excel
26,242 linhas x 74 colunas


,SUBSTÂNCIA,CNPJ,LABORATÓRIO,CÓDIGO GGREM,REGISTRO,EAN 1,EAN 2,EAN 3,PRODUTO,APRESENTAÇÃO,CLASSE TERAPÊUTICA,TIPO DE PRODUTO (STATUS DO PRODUTO),REGIME DE PREÇO,PF Sem Impostos,PF 0%,PF 12 %,PF 12 % ALC,PF 17 %,PF 17 % ALC,"PF 17,5 %","PF 17,5 % ALC",PF 18 %,PF 18 % ALC,PF 19 %,PF 19 % ALC,...,PMC 19 %,PMC 19 % ALC,"PMC 19,5 %","PMC 19,5 % ALC",PMC 20 %,PMC 20 % ALC,"PMC 20,5 %","PMC 20,5 % ALC",PMC 21 %,PMC 21 % ALC,PMC 22 %,PMC 22 % ALC,"PMC 22,5 %","PMC 22,5 % ALC",PMC 23 %,PMC 23 % ALC,RESTRIÇÃO HOSPITALAR,CAP,CONFAZ 87,ICMS 0%,ANÁLISE RECURSAL,LISTA DE CONCESSÃO DE CRÉDITO TRIBUTÁRIO (PIS/COFINS),COMERCIALIZAÇÃO 2025,TARJA,DESTINAÇÃO COMERCIAL
0,21-ACETATO DE DEXAMETASONA;CLOTRIMAZOL,18.459.628/0001-15,BAYER S.A.,538912020009303,1705600230032,7891106000956,-,-,BAYCUTEN N,"10 MG/G + 0,443 MG/G CREM DERM CT BG AL X 40 G",D7B2 - CORTICOESTERÓIDES ASSOCIADOS A ANTIMICO...,Novo,Regulado,"27,44","30,73","34,92","31,18","37,02","33,06","37,25","33,26","37,48","33,46","37,94","33,87",...,"50,90","46,82","51,20","47,11","51,53","47,42","51,85","47,71","52,18","48,01","52,85","48,63","53,19","48,94","53,54","49,26",Não,Não,Não,Não,NaN,Negativa,Sim,- (*),NaN
1,ABATACEPTE,56.998.982/0001-07,BRISTOL-MYERS SQUIBB FARMACÊUTICA LTDA,505107701157215,1018003900019,7896016806469,-,-,ORENCIA,250 MG PO LIOF SOL INJ CT 1 FA + SER DESCARTÁVEL,M1C - AGENTES ANTI-REUMÁTICOS ESPECÍFICOS,Biológico,Regulado,"2098,20","2123,68","2413,27","2384,31","2558,65","2527,95","2574,16","2543,27","2589,85","2558,78","2621,83","2590,37",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sim,Sim,Não,Não,NaN,Positiva,Sim,Tarja Vermelha,NaN
2,ABATACEPTE,56.998.982/0001-07,BRISTOL-MYERS SQUIBB FARMACÊUTICA LTDA,505113100020505,1018003900078,7896016808197,-,-,ORENCIA,125 MG/ML SOL INJ SC CT 4 SER PREENC VD TRANS ...,M1C - AGENTES ANTI-REUMÁTICOS ESPECÍFICOS,Biológico,Regulado,"6662,98","6743,91","7663,54","7571,57","8125,19","8027,69","8174,43","8076,34","8224,28","8125,59","8325,82","8225,90",...,"11476,28","11371,82","11547,55","11442,47","11619,73","11513,97","11692,81","11586,39","11766,81","11659,73","11917,67","11809,20","11994,55","11885,40","12072,43","11962,57",Não,Sim,Sim,Não,NaN,Positiva,Sim,- (*),NaN


## 2. Padronização dos nomes de coluna

Nomes em `snake_case`, sem acentos e sem espaços, para facilitar o uso no código.

In [4]:
def padronizar_nome(nome: str) -> str:
    nome = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    nome = nome.lower().replace("%", "pct")
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")

df = df_raw.rename(columns=padronizar_nome)
df.columns.tolist()

['substancia',
 'cnpj',
 'laboratorio',
 'codigo_ggrem',
 'registro',
 'ean_1',
 'ean_2',
 'ean_3',
 'produto',
 'apresentacao',
 'classe_terapeutica',
 'tipo_de_produto_status_do_produto',
 'regime_de_preco',
 'pf_sem_impostos',
 'pf_0pct',
 'pf_12_pct',
 'pf_12_pct_alc',
 'pf_17_pct',
 'pf_17_pct_alc',
 'pf_17_5_pct',
 'pf_17_5_pct_alc',
 'pf_18_pct',
 'pf_18_pct_alc',
 'pf_19_pct',
 'pf_19_pct_alc',
 'pf_19_5_pct',
 'pf_19_5_pct_alc',
 'pf_20_pct',
 'pf_20_pct_alc',
 'pf_20_5_pct',
 'pf_20_5_pct_alc',
 'pf_21_pct',
 'pf_21_pct_alc',
 'pf_22_pct',
 'pf_22_pct_alc',
 'pf_22_5_pct',
 'pf_22_5_pct_alc',
 'pf_23_pct',
 'pf_23_pct_alc',
 'pmc_sem_impostos',
 'pmc_0_pct',
 'pmc_12_pct',
 'pmc_12_pct_alc',
 'pmc_17_pct',
 'pmc_17_pct_alc',
 'pmc_17_5_pct',
 'pmc_17_5_pct_alc',
 'pmc_18_pct',
 'pmc_18_pct_alc',
 'pmc_19_pct',
 'pmc_19_pct_alc',
 'pmc_19_5_pct',
 'pmc_19_5_pct_alc',
 'pmc_20_pct',
 'pmc_20_pct_alc',
 'pmc_20_5_pct',
 'pmc_20_5_pct_alc',
 'pmc_21_pct',
 'pmc_21_pct_alc',
 'pmc

## 3. Seleção de colunas

Das 36 colunas de preço (PF e PMC em cada alíquota de ICMS), mantemos apenas as versões **sem impostos**
e **ICMS 0%**. Para comparar preços entre medicamentos, o que interessa é o preço sem o efeito da tributação
de cada estado — as demais colunas são derivadas dessas.

In [5]:
colunas = {
    "substancia": "substancia",
    "laboratorio": "laboratorio",
    "cnpj": "cnpj",
    "codigo_ggrem": "codigo_ggrem",
    "registro": "registro",
    "ean_1": "ean",
    "produto": "produto",
    "apresentacao": "apresentacao",
    "classe_terapeutica": "classe_terapeutica",
    "tipo_de_produto_status_do_produto": "tipo_produto",
    "regime_de_preco": "regime_preco",
    "pf_sem_impostos": "pf_sem_impostos",
    "pf_0pct": "pf_icms_0",
    "pmc_sem_impostos": "pmc_sem_impostos",
    "pmc_0_pct": "pmc_icms_0",
    "restricao_hospitalar": "restricao_hospitalar",
    "cap": "cap",
    "confaz_87": "confaz_87",
    "icms_0pct": "isento_icms",
    "analise_recursal": "analise_recursal",
    "lista_de_concessao_de_credito_tributario_pis_cofins": "lista_pis_cofins",
    "comercializacao_2025": "comercializado_ano_anterior",
    "tarja": "tarja",
}

faltando = set(colunas) - set(df.columns)
assert not faltando, f"Colunas não encontradas (layout mudou?): {faltando}"

df = df[list(colunas)].rename(columns=colunas)
df.shape

(26242, 23)

> **Atenção:** a coluna `COMERCIALIZAÇÃO 2025` muda de nome a cada ano (ex.: `COMERCIALIZAÇÃO 2026`).
> Se o `assert` acima falhar em uma lista futura, ajuste essa chave no dicionário.

## 4. Limpeza de textos e valores ausentes

In [6]:
# Remove espaços extras de todas as colunas de texto
df = df.apply(lambda s: s.str.strip())

# Placeholders que representam ausência de informação
PLACEHOLDERS = {"-", "", "nan"}
df = df.replace({p: np.nan for p in PLACEHOLDERS})

# Tarja: "- (*)" = sem informação; demais valores sem o prefixo "Tarja "
df["tarja"] = (
    df["tarja"]
    .replace({"- (*)": np.nan})
    .str.replace(r"^Tarja\s+", "", regex=True)
)

df["tipo_produto"].value_counts(dropna=False)

tipo_produto
Genérico                       9636
Similar                        8823
Novo                           3728
Específico                     2062
Biológico                      1577
Fitoterápico                    320
NaN                              50
Produto de Terapia Avançada      43
Radiofármaco                      3
Name: count, dtype: int64

## 5. Conversão dos preços

Os preços vêm como texto no formato brasileiro (`2098,20`). Convertemos para número,
tratando também separador de milhar e o asterisco que a CMED usa em alguns casos de isenção de ICMS.

In [7]:
COLUNAS_PRECO = ["pf_sem_impostos", "pf_icms_0", "pmc_sem_impostos", "pmc_icms_0"]

def texto_para_numero(s: pd.Series) -> pd.Series:
    limpo = (
        s.str.replace("*", "", regex=False)
         .str.replace(".", "", regex=False)
         .str.replace(",", ".", regex=False)
         .str.strip()
    )
    return pd.to_numeric(limpo, errors="coerce")

for col in COLUNAS_PRECO:
    antes = df[col].notna().sum()
    df[col] = texto_para_numero(df[col])
    perdidos = antes - df[col].notna().sum()
    print(f"{col:18} convertidos: {df[col].notna().sum():>6,}  | falharam: {perdidos}")

df[COLUNAS_PRECO].describe().T

pf_sem_impostos    convertidos: 26,242  | falharam: 0
pf_icms_0          convertidos: 26,242  | falharam: 0
pmc_sem_impostos   convertidos: 22,318  | falharam: 0
pmc_icms_0         convertidos: 22,318  | falharam: 0


,count,mean,std,min,25%,50%,75%,max
pf_sem_impostos,26242.0,12853.962275,291491.113999,0.48,31.2200,86.65,294.7525,7733930.44
pf_icms_0,26242.0,14330.364363,326465.346023,0.49,33.0100,91.79,310.7300,8662003.48
pmc_sem_impostos,22318.0,1744.533356,24724.518195,1.68,38.2025,93.61,266.7025,3142953.89
pmc_icms_0,22318.0,1860.111529,26850.386673,1.70,39.7500,96.85,274.7250,3415769.38


O PMC fica vazio em cerca de 3,9 mil apresentações. Isso é esperado: segundo as notas da própria lista,
medicamentos de **uso restrito a hospitais** não podem ser vendidos pelo PMC, então esse preço é omitido.

In [8]:
pd.crosstab(df["restricao_hospitalar"], df["pmc_sem_impostos"].isna(), 
            rownames=["restrição hospitalar"], colnames=["PMC vazio"])

PMC vazio,False,True
restrição hospitalar,,
Não,22311,0
Sim,7,3924


## 6. Campos Sim/Não → booleanos

In [9]:
COLUNAS_BOOL = ["restricao_hospitalar", "cap", "confaz_87", "isento_icms", "comercializado_ano_anterior"]

for col in COLUNAS_BOOL:
    df[col] = df[col].map({"Sim": True, "Não": False}).astype("boolean")

df["em_analise_recursal"] = df["analise_recursal"].notna()
df = df.drop(columns="analise_recursal")

df[COLUNAS_BOOL].mean().round(3)

restricao_hospitalar            0.15
cap                            0.105
confaz_87                      0.112
isento_icms                    0.044
comercializado_ano_anterior    0.496
dtype: Float64

## 7. Novas colunas (features)

- **`classe_codigo` / `classe_descricao`:** separa o código (ex.: `M1C`) da descrição da classe terapêutica.
- **`grupo_anatomico`:** a primeira letra do código indica o grupo anatômico principal (ex.: `M` = sistema musculoesquelético).
- **`n_substancias`:** quantidade de princípios ativos (associações vêm separadas por `;`).
- **`concorrentes_substancia`:** quantos laboratórios diferentes vendem a mesma substância — uma medida simples de concorrência.

In [10]:
partes = df["classe_terapeutica"].str.split(" - ", n=1, expand=True)
df["classe_codigo"] = partes[0].str.strip()
df["classe_descricao"] = partes[1].str.strip()
df["grupo_anatomico"] = df["classe_codigo"].str[0]

df["n_substancias"] = df["substancia"].str.count(";") + 1

df["concorrentes_substancia"] = df.groupby("substancia")["laboratorio"].transform("nunique")

df[["substancia", "classe_codigo", "grupo_anatomico", "n_substancias", "concorrentes_substancia"]].sample(5, random_state=42)

,substancia,classe_codigo,grupo_anatomico,n_substancias,concorrentes_substancia
24603,TEICOPLANINA,J1X1,J,1,7
4343,CENOBAMATO,N3A,N,1,1
8364,CLORIDRATO DE NEBIVOLOL,C7A0,C,1,12
26016,ÁCIDO FÓLICO,B3X,B,1,12
2516,BESILATO DE LEVANLODIPINO,C8A,C,1,3


## 8. Checagens de qualidade

In [11]:
assert df["codigo_ggrem"].is_unique, "Código GGREM duplicado"
assert (df["pf_sem_impostos"] > 0).all(), "Existe PF zerado ou negativo"

# O PMC deve ser sempre maior ou igual ao PF (margem do varejo)
com_pmc = df.dropna(subset=["pmc_sem_impostos"])
inconsistentes = com_pmc[com_pmc["pmc_sem_impostos"] < com_pmc["pf_sem_impostos"]]
print(f"Apresentações com PMC < PF: {len(inconsistentes)}")

df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)

Apresentações com PMC < PF: 0


tarja               4670
pmc_sem_impostos    3924
pmc_icms_0          3924
tipo_produto          50
dtype: int64

## 9. Salvando o resultado

Guardamos a data de publicação da lista (extraída do nome do arquivo) para permitir,
no futuro, juntar várias edições e analisar os reajustes ao longo do tempo.

In [12]:
data_str = re.search(r"(\d{8})", ARQUIVO.name).group(1)
df["data_lista"] = pd.to_datetime(data_str, format="%Y%m%d")

saida = PASTA_PROCESSED / f"cmed_limpo_{data_str}.csv"
df.to_csv(saida, index=False, encoding="utf-8")
print(f"Salvo em {saida}  ->  {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Salvo em ..\data\processed\cmed_limpo_20260909.csv  ->  26,242 linhas x 29 colunas


## Resumo

- Base com todas as apresentações da lista, uma linha por código GGREM.
- Preços convertidos para número, mantendo apenas as versões sem impostos e com ICMS 0%.
- PMC vazio apenas em produtos de uso hospitalar, como previsto pela regulação.
- Cerca de metade das apresentações **não foi comercializada** no ano anterior. Para as análises de mercado,
  o próximo notebook vai filtrar `comercializado_ano_anterior == True`.

**Próximo passo:** `02_eda.ipynb` — diferenças de preço entre genéricos, similares e medicamentos novos.